# 68. The AS/RS Task Interleaving Problem
## Tier 5: The Integrated Digital Twin (System-of-Systems Simulation)

### Key Assumptions
- Four-layer digital twin architecture: Physical Assets → Connectivity → Data Processing → Application
- Real-time synchronization between physical and virtual AS/RS systems with 1-second update cycles
- IoT sensor network with 500 sensors and 50 RFID readers for continuous monitoring
- Physics-based simulation of crane dynamics, storage rack states, and task queues
- Predictive analytics for demand forecasting and congestion prediction
- What-if scenario analysis with 10x real-time testing capability

### Approach (Step-by-Step)
1. **Physical Layer Modeling**: Simulate crane movements, storage states, and sensor data
2. **Connectivity Layer**: Implement IoT communication and data streaming protocols
3. **Data Processing Layer**: Real-time data fusion and state estimation
4. **Application Layer**: Task optimization algorithms and visualization dashboards
5. **Scenario Testing**: Multi-scenario analysis with disruption modeling

### What to Look for in the Results
- Real-time synchronization accuracy between physical and digital systems
- Predictive analytics performance for demand forecasting
- What-if scenario analysis results for different operational conditions
- System resilience metrics under equipment failures and maintenance windows
- KPI monitoring and performance optimization recommendations

### Concrete Example (from the source)
Comprehensive digital twin simulation with three operational scenarios:

**Scenario 1 - Peak Operations**: 200 tasks/hour, 3 active cranes
- Traditional Sequential Processing: 156 tasks completed, 28% idle time
- AI-Optimized Interleaving: 187 tasks completed, 12% idle time
- **Performance Improvement**: 20% throughput increase

**Scenario 2 - Maintenance Mode**: 1 crane offline, 150 tasks/hour
- Traditional Processing: 89 tasks completed, significant backlog
- Adaptive Interleaving: 134 tasks completed, managed workload
- **Resilience Improvement**: 51% better performance under constraints

**Scenario 3 - Rush Order Integration**: 50 priority tasks inserted mid-shift
- Static Algorithms: 23-minute average delay for regular tasks
- Dynamic Re-optimization: 7-minute average delay
- **Responsiveness Improvement**: 70% faster adaptation to disruptions

In [1]:
# Import required libraries for Digital Twin implementation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up professional visualization style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
@dataclass
class Task:
    """Represents a storage or retrieval task in AS/RS."""
    id: str
    task_type: str  # 'storage' or 'retrieval'
    x: int
    y: int
    priority: int
    deadline: Optional[float] = None
    created_time: float = 0.0
    assigned_time: Optional[float] = None
    completed_time: Optional[float] = None

@dataclass
class Crane:
    """Represents an AS/RS crane with position and state."""
    id: str
    x: int
    y: int
    battery_level: float  # 0-100%
    status: str  # 'idle', 'moving', 'loading', 'unloading', 'maintenance'
    current_task: Optional[Task] = None
    speed: float = 1.0  # cells per second
    max_capacity: int = 1

@dataclass
class Sensor:
    """IoT sensor for monitoring AS/RS state."""
    id: str
    sensor_type: str  # 'position', 'load', 'battery', 'temperature'
    location_x: int
    location_y: int
    value: float
    timestamp: float
    accuracy: float = 0.95

@dataclass
class DigitalTwinState:
    """Current state of the digital twin system."""
    current_time: float
    tasks: List[Task]
    cranes: List[Crane]
    sensors: List[Sensor]
    completed_tasks: List[Task]
    pending_tasks: List[Task]
    system_metrics: Dict[str, float]
    predictions: Dict[str, float]
    alerts: List[Dict[str, str]]

In [ ]:
class ASRSDigitalTwin:
    """
    Integrated Digital Twin for AS/RS Task Interleaving.
    """
    
    def __init__(self, num_crane=3, num_aisles=6, aisle_length=20, aisle_height=10):
        """
        Initialize digital twin.
        
        Args:
            num_crane: Number of cranes in the system
            num_aisles: Number of AS/RS aisles
            aisle_length: Length of each aisle
            aisle_height: Height of each aisle
        """
        self.num_crane = num_crane
        self.num_aisles = num_aisles
        self.aisle_length = aisle_length
        self.aisle_height = aisle_height
        
        # Initialize system state
        self.state = DigitalTwinState(
            current_time=0.0,
            tasks=[],
            cranes=[],
            sensors=[],
            completed_tasks=[],
            pending_tasks=[],
            system_metrics={},
            predictions={},
            alerts=[]
        )
        
        # Initialize cranes
        self._init_cranes()
        
        # Initialize sensors
        self._init_sensors()
        
        # Tracking variables
        self.update_history = []
        self.task_completion_times = []
        self.crane_utilization = defaultdict(list)
        
        # Performance metrics
        self.total_tasks_processed = 0
        self.total_travel_distance = 0
        self.total_idle_time = 0
        self.total_processing_time = 0
    
    def _init_cranes(self):
        """Initialize crane fleet."""
        for i in range(self.num_crane):
            crane = Crane(
                id=f"CR{i+1}",
                x=1,
                y=1,
                battery_level=100.0,
                status='idle',
                speed=2.0,
                max_capacity=1
            )
            self.state.cranes.append(crane)
    
    def _init_sensors(self):
        """Initialize IoT sensor network."""
        sensor_id = 0
        
        # Position sensors for each crane
        for crane in self.state.cranes:
            sensor_id += 1
            pos_sensor = Sensor(
                id=f"POS_{crane.id}",
                sensor_type='position',
                location_x=crane.x,
                location_y=crane.y,
                value=0.0,
                timestamp=0.0,
                accuracy=0.98
            )
            self.state.sensors.append(pos_sensor)
            
            # Battery sensor
            sensor_id += 1
            battery_sensor = Sensor(
                id=f"BAT_{crane.id}",
                sensor_type='battery',
                location_x=crane.x,
                location_y=crane.y,
                value=crane.battery_level,
                timestamp=0.0,
                accuracy=0.99
            )
            self.state.sensors.append(battery_sensor)
        
        # Task queue sensors
        for aisle in range(self.num_aisles):
            for position in range(self.aisle_length * self.aisle_height):
                sensor_id += 1
                load_sensor = Sensor(
                    id=f"LOAD_A{aisle+1}_P{position+1}",
                    sensor_type='load',
                    location_x=position % self.aisle_length + 1,
                    location_y=position // self.aisle_length + 1,
                    value=0.0,
                    timestamp=0.0,
                    accuracy=0.95
                )
                self.state.sensors.append(load_sensor)
        
        # Environmental sensors
        for i in range(10):  # Temperature sensors
            sensor_id += 1
            temp_sensor = Sensor(
                id=f"TEMP_{i+1}",
                sensor_type='temperature',
                location_x=random.randint(1, self.aisle_length),
                location_y=random.randint(1, self.aisle_height),
                value=20.0 + random.gauss(0, 2),
                timestamp=0.0,
                accuracy=0.90
            )
            self.state.sensors.append(temp_sensor)
    
    def calculate_distance(self, pos1, pos2):
        """Calculate Manhattan distance between two positions."""
        return max(abs(pos2[0] - pos1[0]), abs(pos2[1] - pos1[1]))
    
    def calculate_task_time(self, task, from_pos, to_pos):
        """Calculate time to complete a task between positions."""
        # Travel time
        travel_time = self.calculate_distance(from_pos, to_pos) * 0.5
        
        # Operation time
        operation_time = 3.0 if task.task_type == 'storage' else 2.0
        
        return travel_time + operation_time
    
    def add_task(self, task_id, task_type, x, y, priority, deadline=None):
        """Add a new task to the system."""
        task = Task(
            id=task_id,
            task_type=task_type,
            x=x,
            y=y,
            priority=priority,
            deadline=deadline,
            created_time=self.state.current_time
        )
        
        self.state.tasks.append(task)
        self.state.pending_tasks.append(task)
        
        # Generate alert for high priority tasks
        if priority >= 8:
            self.state.alerts.append({
                'timestamp': self.state.current_time,
                'type': 'high_priority_task',
                'message': f'High priority task {task_id} added',
                'task_id': task_id
            })
    
    def update_sensors(self):
        """Update all sensor readings."""
        for sensor in self.state.sensors:
            sensor.timestamp = self.state.current_time
            
            # Update sensor values based on current state
            if sensor.sensor_type == 'position':
                # Find crane with this position sensor
                for crane in self.state.cranes:
                    if crane.id == sensor.id.replace('POS_', ''):
                        sensor.value = crane.x * 100 + crane.y  # Encoded position
                        break
            
            elif sensor.sensor_type == 'battery':
                # Find crane with this battery sensor
                for crane in self.state.cranes:
                    if crane.id == sensor.id.replace('BAT_', ''):
                        sensor.value = crane.battery_level
                        break
            
            elif sensor.sensor_type == 'load':
                # Check if location is occupied
                occupied = False
                for task in self.state.completed_tasks + self.state.pending_tasks:
                    if task.x == sensor.location_x and task.y == sensor.location_y:
                        occupied = True
                        break
                sensor.value = 1.0 if occupied else 0.0
            
            elif sensor.sensor_type == 'temperature':
                # Simulate temperature variations
                base_temp = 20.0
                time_factor = (self.state.current_time % 3600) / 3600  # Daily cycle
                sensor.value = base_temp + 5 * np.sin(time_factor * 2 * np.pi)
    
    def predict_demand(self):
        """Predict future demand based on historical patterns."""
        # Simple demand prediction based on time of day and current workload
        hour_of_day = (self.state.current_time % 86400) / 3600
        
        # Base demand patterns
        if 6 <= hour_of_day <= 10:  # Morning peak
            base_demand = 2.0
        elif 16 <= hour_of_day <= 18:  # Evening peak
            base_demand = 2.5
        else:  # Off-peak
            base_demand = 1.0
        
        # Adjust based on current workload
        current_load = len(self.state.pending_tasks)
        max_capacity = self.num_crane * 10  # Max tasks per hour
        load_factor = min(current_load / max_capacity, 1.5)
        
        predicted_demand = base_demand * load_factor
        
        self.state.predictions['predicted_demand'] = predicted_demand
        self.state.predictions['prediction_confidence'] = 0.85
    
    def predict_congestion(self):
        """Predict system congestion based on current state."""
        # Calculate congestion metrics
        crane_utilization = sum(1 for crane in self.state.cranes if crane.status != 'idle') / self.num_crane
        task_queue_ratio = len(self.state.pending_tasks) / max(len(self.state.tasks), 1)
        
        # Predict congestion probability
        congestion_prob = 0.3 * crane_utilization + 0.7 * task_queue_ratio
        
        self.state.predictions['congestion_probability'] = congestion_prob
        self.state.predictions['congestion_confidence'] = 0.90
    
    def assign_task_to_crane(self, task, crane):
        """Assign a task to a crane."""
        task.assigned_time = self.state.current_time
        crane.current_task = task
        crane.status = 'moving'
        
        # Remove from pending tasks
        if task in self.state.pending_tasks:
            self.state.pending_tasks.remove(task)
    
    def complete_task(self, task, crane):
        """Complete a task and update crane state."""
        task.completed_time = self.state.current_time
        crane.current_task = None
        crane.status = 'idle'
        
        # Move to completed tasks
        if task in self.state.tasks:
            self.state.tasks.remove(task)
            self.state.completed_tasks.append(task)
        
        # Update metrics
        completion_time = task.completed_time - task.created_time
        self.task_completion_times.append(completion_time)
        
        # Generate completion alert if task is late
        if task.deadline and task.completed_time > task.deadline:
            self.state.alerts.append({
                'timestamp': self.state.current_time,
                'type': 'late_task',
                'message': f'Task {task.id} completed {completion_time - task.deadline:.1f}s late',
                'task_id': task.id
            })
    
    def simulate_crane_movement(self, crane, target_pos, dt):
        """Simulate crane movement towards target position."""
        current_pos = (crane.x, crane.y)
        distance = self.calculate_distance(current_pos, target_pos)
        
        if distance > 0:
            # Calculate movement vector
            dx = (target_pos[0] - current_pos[0]) / distance
            dy = (target_pos[1] - current_pos[1]) / distance
            
            # Move crane
            movement = min(crane.speed * dt, distance)
            crane.x += int(movement * dx)
            crane.y += int(movement * dy)
            
            # Update battery
            battery_consumption = movement * 0.1  # 0.1% per unit movement
            crane.battery_level = max(0, crane.battery_level - battery_consumption)
            
            # Check if reached target
            if self.calculate_distance((crane.x, crane.y), target_pos) < 1:
                return True
        
        return False
    
    def process_task(self, crane, dt):
        """Process current task assigned to crane."""
        task = crane.current_task
        if not task:
            return
        
        # Move to task location
        task_pos = (task.x, task.y)
        crane_pos = (crane.x, crane.y)
        
        # Simulate movement
        if self.simulate_crane_movement(crane, task_pos, dt):
            # Process task at location
            crane.status = 'loading' if task.task_type == 'storage' else 'unloading'
            
            # Simulate processing time
            processing_time = 3.0 if task.task_type == 'storage' else 2.0
            
            # Update time
            self.state.current_time += processing_time
            
            # Complete task
            self.complete_task(task, crane)
            
            # Return to depot
            depot_pos = (1, 1)
            self.simulate_crane_movement(crane, depot_pos, dt)
        else:
            # Still moving to task
            pass
    
    def find_best_crane_for_task(self, task):
        """Find the best crane for a task based on multiple criteria."""
        available_cranes = [c for c in self.state.cranes if c.status == 'idle' and c.battery_level > 20]
        
        if not available_cranes:
            return None
        
        best_crane = None
        best_score = float('inf')
        
        for crane in available_cranes:
            # Calculate distance-based score
            distance = self.calculate_distance((crane.x, crane.y), (task.x, task.y))
            distance_score = distance / 10.0
            
            # Calculate battery-based score
            battery_score = (100 - crane.battery_level) / 100.0
            
            # Calculate priority-based score
            priority_score = (10 - task.priority) / 10.0
            
            # Combined score (lower is better)
            total_score = 0.4 * distance_score + 0.3 * battery_score + 0.3 * priority_score
            
            if total_score < best_score:
                best_score = total_score
                best_crane = crane
        
        return best_crane
    
    def optimize_task_assignment(self):
        """Optimize task assignment using greedy algorithm with predictions."""
        # Sort pending tasks by priority (descending)
        sorted_tasks = sorted(self.state.pending_tasks, 
                           key=lambda t: (t.priority, t.deadline or float('inf'), t.created_time), 
                           reverse=True)
        
        # Assign tasks to cranes
        for task in sorted_tasks:
            crane = self.find_best_crane_for_task(task)
            if crane:
                self.assign_task_to_crane(task, crane)
            else:
                # No available cranes, wait
                break
    
    def update_system_metrics(self):
        """Update system performance metrics."""
        # Basic metrics
        self.state.system_metrics['total_tasks'] = len(self.state.tasks) + len(self.state.completed_tasks)
        self.state.system_metrics['pending_tasks'] = len(self.state.pending_tasks)
        self.state.system_metrics['completed_tasks'] = len(self.state.completed_tasks)
        self.state.system_metrics['active_cranes'] = sum(1 for c in self.state.cranes if c.status != 'idle')
        
        # Performance metrics
        if self.task_completion_times:
            self.state.system_metrics['avg_completion_time'] = np.mean(self.task_completion_times)
            self.state.system_metrics['completion_time_std'] = np.std(self.task_completion_times)
        else:
            self.state.system_metrics['avg_completion_time'] = 0
            self.state.system_metrics['completion_time_std'] = 0
        
        # Utilization metrics
        for crane in self.state.cranes:
            utilization = sum(1 for t in self.task_completion_times if 
                          t.created_time >= self.state.current_time - 3600) / 3600) / 3600
            self.crane_utilization[crane.id] = utilization
        
        # Alert metrics
        self.state.system_metrics['active_alerts'] = len([a for a in self.state.alerts if a['timestamp'] > self.state.current_time - 300])
        
        # Prediction metrics
        if 'predicted_demand' in self.state.predictions:
            self.state.system_metrics['predicted_demand'] = self.state.predictions['predicted_demand']
        if 'congestion_probability' in self.state.predictions:
            self.state.system_metrics['congestion_probability'] = self.state.predictions['congestion_probability']
    
    def step(self, dt=1.0):
        """Advance simulation by dt seconds."""
        # Update time
        self.state.current_time += dt
        
        # Update sensors
        self.update_sensors()
        
        # Make predictions
        self.predict_demand()
        self.predict_congestion()
        
        # Optimize task assignment
        self.optimize_task_assignment()
        
        # Process crane operations
        for crane in self.state.cranes:
            self.process_task(crane, dt)
        
        # Update metrics
        self.update_system_metrics()
        
        # Store update history
        self.update_history.append({
            'timestamp': self.state.current_time,
            'pending_tasks': len(self.state.pending_tasks),
            'completed_tasks': len(self.state.completed_tasks),
            'active_cranes': sum(1 for c in self.state.cranes if c.status != 'idle'),
            'avg_completion_time': self.state.system_metrics.get('avg_completion_time', 0),
            'predicted_demand': self.state.predictions.get('predicted_demand', 0),
            'congestion_probability': self.state.predictions.get('congestion_probability', 0)
        })
    
    def run_scenario(self, duration_hours=8.0, task_rate=1.0, disruption_type=None):
        """Run a complete simulation scenario."""
        print(f"Running scenario: {disruption_type or 'Normal'}")
        print(f"Duration: {duration_hours} hours")
        print(f"Task rate: {task_rate} tasks/minute")
        print("=" * 60)
        
        # Reset state
        self.state = DigitalTwinState(
            current_time=0.0,
            tasks=[],
            cranes=self.state.cranes.copy(),
            sensors=self.state.sensors.copy(),
            completed_tasks=[],
            pending_tasks=[],
            system_metrics={},
            predictions={},
            alerts=[]
        )
        
        # Reset cranes
        for crane in self.state.cranes:
            crane.status = 'idle'
            crane.battery_level = 100.0
            crane.current_task = None
        
        # Simulation parameters
        total_duration = duration_hours * 3600  # Convert to seconds
        task_interval = 60.0 / task_rate  # Seconds between tasks
        next_task_time = 0.0
        task_counter = 0
        
        # Add disruption if specified
        if disruption_type == 'crane_failure':
            # Random crane failure
            failed_crane = random.choice(self.state.cranes)
            failed_crane.status = 'maintenance'
            failed_crane.battery_level = 10.0
            print(f"\nDISRUPTION: Crane {failed_crane.id} failed at time {self.state.current_time:.1f}s")
        
        elif disruption_type == 'rush_orders':
            # Add rush orders
            rush_tasks = 20
            
            for i in range(rush_tasks):
                task_id = f"RUSH_{i+1}"
                task_type = random.choice(['storage', 'retrieval'])
                x = random.randint(1, self.aisle_length)
                y = random.randint(1, self.aisle_height)
                priority = random.randint(8, 10)
                deadline = self.state.current_time + random.uniform(300, 600)  # 5-10 minutes
                
                self.add_task(task_id, task_type, x, y, priority, deadline)
            print(f"\nDISRUPTION: {rush_tasks} rush orders added")
        
        # Main simulation loop
        start_time = time.time()
        while self.state.current_time < total_duration:
            # Add tasks at specified rate
            if self.state.current_time >= next_task_time and len(self.state.tasks) < 1000:  # Limit total tasks
                task_counter += 1
                task_id = f"T{task_counter}"
                task_type = random.choice(['storage', 'retrieval'])
                x = random.randint(1, self.aisle_length)
                y = random.randint(1, self.aisle_height)
                priority = random.randint(1, 10)
                
                self.add_task(task_id, task_type, x, y, priority)
                
                next_task_time += task_interval
            
            # Step simulation
            self.step(dt=1.0)
            
            # Progress reporting
            if int(self.state.current_time) % 600 == 0:  # Every 10 minutes
                elapsed = self.state.current_time
                completed = len(self.state.completed_tasks)
                pending = len(self.state.pending_tasks)
                active_cranes = sum(1 for c in self.state.cranes if c.status != 'idle')
                
                print(f"Time: {elapsed/3600:.1f}h | Tasks: {completed}/{completed+pending} | "
                      f"Active Cranes: {active_cranes}/{self.num_crane} | "
                      f"Demand: {self.state.predictions.get('predicted_demand', 0):.2f}/h | ")
        
        end_time = time.time()
        total_time = end_time - start_time
        
        # Final statistics
        print(f"\nScenario completed in {total_time:.2f} seconds")
        print(f"Total tasks processed: {len(self.state.completed_tasks)}")
        print(f"Average completion time: {self.state.system_metrics.get('avg_completion_time', 0):.2f}s")
        
        # Return results
        return {
            'total_time': total_time,
            'tasks_completed': len(self.state.completed_tasks),
            'avg_completion_time': self.state.system_metrics.get('avg_completion_time', 0),
            'final_pending': len(self.state.pending_tasks),
            'alerts_generated': len(self.state.alerts),
            'crane_utilization': self.crane_utilization
        }
    
    def analyze_scenario_results(self, results):
        """Analyze and visualize scenario results."""
        print(f"\nScenario Analysis:")
        print(f"Total simulation time: {results['total_time']:.2f} seconds")
        print(f"Tasks completed: {results['tasks_completed']}")
        print(f"Average completion time: {results['avg_completion_time']:.2f} seconds")
        print(f"Final pending tasks: {results['final_pending']}")
        print(f"Alerts generated: {results['alerts_generated']}")
        
        # Crane utilization analysis
        print(f"\nCrane Utilization:")
        for crane_id, utilization in results['crane_utilization'].items():
            print(f"  {crane_id}: {utilization:.1%}")
        avg_utilization = np.mean(list(results['crane_utilization'].values()))
        print(f"  Average: {avg_utilization:.1%}")
        
        # Alert analysis
        if results['alerts_generated'] > 0:
            print(f"\nAlert Summary:")
            alert_types = defaultdict(int)
            for alert in self.state.alerts:
                alert_types[alert['type']] += 1
            for alert_type, count in alert_types.items():
                print(f"  {alert_type}: {count}")
        else:
            print(f"\nNo alerts generated during simulation")
        
        # Performance metrics
        throughput = results['tasks_completed'] / (results['total_time'] / 3600)
        print(f"\nPerformance Metrics:")
        print(f"  Throughput: {throughput:.2f} tasks/hour")
        print(f"  Efficiency: {results['tasks_completed'] / max(1, results['tasks_completed'] + results['final_pending']):.1%}")
        
        return results

In [ ]:
# Initialize the Digital Twin system
print("AS/RS Task Interleaving - Digital Twin Simulation")
print(f"Number of cranes: 3")
print(f"Number of aisles: 6")
print(f"Aisle dimensions: 20x10")
print()

# Create digital twin instance
digital_twin = ASRSDigitalTwin(num_crane=3, num_aisles=6, aisle_length=20, aisle_height=10)

print("Digital Twin initialized successfully!")
print(f"Active cranes: {len([c for c in digital_twin.state.cranes if c.status != 'idle'])}")
print(f"Total sensors: {len(digital_twin.state.sensors)}")
print(f"System time: {digital_twin.state.current_time:.1f}s")

In [ ]:
# Scenario 1: Normal Operations
print("\n" + "=" * 60)
print("SCENARIO 1: NORMAL OPERATIONS")
print("=" * 60)

normal_results = digital_twin.run_scenario(
    duration_hours=8.0,
    task_rate=1.0,
    disruption_type=None
)

normal_analysis = digital_twin.analyze_scenario_results(normal_results)

In [ ]:
# Scenario 2: Peak Operations
print("\n" + "=" * 60)
print("SCENARIO 2: PEAK OPERATIONS")
print("=" * 60)

peak_results = digital_twin.run_scenario(
    duration_hours=8.0,
    task_rate=2.0,  # Double task rate
    disruption_type=None
)

peak_analysis = digital_twin.analyze_scenario_results(peak_results)

# Comparison with normal operations
print(f"\nPeak vs Normal Comparison:")
improvement = ((peak_results['tasks_completed'] - normal_results['tasks_completed']) / normal_results['tasks_completed']) * 100
print(f"Tasks completed: {normal_results['tasks_completed']} → {peak_results['tasks_completed']} ({improvement:.1f}% improvement)")
print(f"Throughput: {normal_results['tasks_completed'] / 8:.1f} → {peak_results['tasks_completed'] / 8:.1f} tasks/hour")

In [ ]:
# Scenario 3: Maintenance Mode
print("\n" + "=" * 60)
print("SCENARIO 3: MAINTENANCE MODE")
print("=" * 60)

maintenance_results = digital_twin.run_scenario(
    duration_hours=8.0,
    task_rate=1.5,
    disruption_type='crane_failure'
)

maintenance_analysis = digital_twin.analyze_scenario_results(maintenance_results)

# Comparison with normal operations
print(f"\nMaintenance vs Normal Comparison:")
degradation = ((normal_results['tasks_completed'] - maintenance_results['tasks_completed']) / normal_results['tasks_completed']) * 100
print(f"Tasks completed: {normal_results['tasks_completed']} → {maintenance_results['tasks_completed']} ({degradation:.1f}% degradation)")
print(f"Throughput: {normal_results['tasks_completed'] / 8:.1f} → {maintenance_results['tasks_completed'] / 8:.1f} tasks/hour")

In [ ]:
# Scenario 4: Rush Orders
print("\n" + "=" * 60)
print("SCENARIO 4: RUSH ORDERS")
print("=" * 60)

rush_results = digital_twin.run_scenario(
    duration_hours=8.0,
    task_rate=1.5,
    disruption_type='rush_orders'
)

rush_analysis = digital_twin.analyze_scenario_results(rush_results)

# Comparison with normal operations
print(f"\nRush Orders vs Normal Comparison:")
rush_improvement = ((rush_results['tasks_completed'] - normal_results['tasks_completed']) / normal_results['tasks_completed']) * 100
print(f"Tasks completed: {normal_results['tasks_completed']} → {rush_results['tasks_completed']} ({rush_improvement:.1f}% improvement)")
print(f"Throughput: {normal_results['tasks_completed'] / 8:.1f} → {rush_results['tasks_completed'] / 8:.1f} tasks/hour")

# Alert analysis for rush orders
print(f"\nRush Order Alerts:")
rush_alerts = [a for a in digital_twin.state.alerts if a['timestamp'] > rush_results['total_time'] - 300]
print(f"High priority alerts generated: {len(rush_alerts)}")

In [ ]:
# Visualize comprehensive results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Task completion over time
normal_times = [h['timestamp'] for h in digital_twin.update_history]
normal_completed = [h['completed_tasks'] for h in digital_twin.update_history]
peak_times = [h['timestamp'] for h in digital_twin.update_history if 'peak' in str(h.get('scenario', ''))]
peak_completed = [h['completed_tasks'] for h in digital_twin.update_history if 'peak' in str(h.get('scenario', ''))]
maintenance_times = [h['timestamp'] for h in digital_twin.update_history if 'maintenance' in str(h.get('scenario', ''))]
maintenance_completed = [h['completed_tasks'] for h in digital_twin.update_history if 'maintenance' in str(h.get('scenario', ''))]
rush_times = [h['timestamp'] for h in digital_twin.update_history if 'rush' in str(h.get('scenario', ''))]
rush_completed = [h['completed_tasks'] for h in digital_twin.update_history if 'rush' in str(h.get('scenario', ''))]

ax1.plot(normal_times, normal_completed, 'b-', linewidth=2, label='Normal', alpha=0.7)
if peak_times:
    ax1.plot(peak_times, peak_completed, 'r-', linewidth=2, label='Peak', alpha=0.7)
if maintenance_times:
    ax1.plot(maintenance_times, maintenance_completed, 'orange', linewidth=2, label='Maintenance', alpha=0.7)
if rush_times:
    ax1.plot(rush_times, rush_completed, 'purple', linewidth=2, label='Rush Orders', alpha=0.7)

ax1.set_xlabel('Time (hours)')
ax1.set_ylabel('Completed Tasks')
ax1.set_title('Task Completion Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: System performance comparison
scenarios = ['Normal', 'Peak', 'Maintenance', 'Rush Orders']
completed_counts = [
    normal_results['tasks_completed'],
    peak_results['tasks_completed'],
    maintenance_results['tasks_completed'],
    rush_results['tasks_completed']
]

bars = ax2.bar(scenarios, completed_counts, color=['blue', 'red', 'orange', 'purple'], alpha=0.7)
ax2.set_ylabel('Tasks Completed')
ax2.set_title('Scenario Performance Comparison')
ax2.grid(True, alpha=0.3)

# Add value labels on bars
for bar, count in zip(bars, completed_counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{count}', ha='center', va='bottom')

# Plot 3: Alert frequency analysis
alert_timeline = [h['timestamp'] for h in digital_twin.update_history if h.get('alerts', [])]
alert_counts = []
alert_types = ['high_priority_task', 'late_task']

for i, alert_type in enumerate(alert_types):
    type_alerts = [h['timestamp'] for h in digital_twin.update_history if h.get('alerts', []) if h['alerts'].get(alert_type, 0) > 0]
    alert_counts.append(len(type_alerts))

ax3.bar(alert_types, alert_counts, color=['red', 'orange'], alpha=0.7)
ax3.set_xlabel('Alert Type')
ax3.set_ylabel('Alert Count')
ax3.set_title('Alert Frequency Analysis')
ax3.grid(True, alpha=0.3)

# Plot 4: Crane utilization over time
crane_colors = plt.cm.Set3(np.linspace(0, 1, len(digital_twin.state.cranes)))

for i, crane in enumerate(digital_twin.state.cranes):
    utilization = digital_twin.crane_utilization.get(crane.id, [])
    if utilization:
        ax4.plot(normal_times, utilization, color=[crane_colors[i]], 
                 linewidth=2, label=f'Crane {crane.id}', alpha=0.7)

ax4.set_xlabel('Time (hours)')
ax4.set_ylabel('Utilization (%)')
ax4.set_title('Crane Utilization Over Time')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nDigital Twin Performance Summary:")
print(f"Normal operations: {normal_results['tasks_completed']} tasks completed")
print(f"Peak operations: {peak_results['tasks_completed']} tasks completed")
print(f"Maintenance mode: {maintenance_results['tasks_completed']} tasks completed")
print(f"Rush orders: {rush_results['tasks_completed']} tasks completed")
print(f"\nTotal alerts: {len(digital_twin.state.alerts)}")
print(f"Average throughput: {normal_results['tasks_completed'] / 8:.1f} tasks/hour")

In [ ]:
# Generate detailed performance report
print("\n" + "=" * 80)
print("DIGITAL TWIN PERFORMANCE REPORT")
print("=" * 80)

# System configuration
print("System Configuration:")
print(f"  Cranes: {len(digital_twin.state.cranes)}")
print(f"  Aisles: {digital_twin.num_aisles}")
print(f"  Storage positions: {digital_twin.num_aisles * digital_twin.aisle_length * digital_twin.aisle_height}")
print(f"  Sensors: {len(digital_twin.state.sensors)}")
print(f"  Update rate: 1 Hz")

# Scenario comparison table
print("\nScenario Comparison Table:")
print("| Scenario | Tasks Completed | Throughput | Efficiency | Alerts |")
print("|---------|-----------------|-----------|----------|-------|")

scenarios = [
    ('Normal', normal_results),
    ('Peak', peak_results),
    ('Maintenance', maintenance_results),
    ('Rush Orders', rush_results)
]

for scenario_name, results in scenarios:
    throughput = results['tasks_completed'] / 8.0  # tasks per hour
    efficiency = results['tasks_completed'] / max(1, results['tasks_completed'] + results['final_pending'])
    alerts = len([a for a in digital_twin.state.alerts if a['timestamp'] > results['total_time'] - 300])
    
    print(f"| {scenario_name:13s} | {results['tasks_completed']:16d} | {throughput:7.2f}/h | {efficiency:7.2%} | {alerts:4d} |")

# Key insights
print(f"\nKey Insights:")
print(f"• Peak operations increase throughput by {(peak_results['tasks_completed'] - normal_results['tasks_completed']) / normal_results['tasks_completed'] * 100:.1f}%")
print(f"• Maintenance mode reduces efficiency by {(normal_results['tasks_completed'] - maintenance_results['tasks_completed']) / normal_results['tasks_completed'] * 100:.1f}%")
print(f"• Rush orders increase efficiency by {(rush_results['tasks_completed'] - normal_results['tasks_completed']) / normal_results['tasks_completed'] * 100:.1f}%")

# Resilience metrics
print(f"\nResilience Analysis:")
print(f"• Crane failure resilience: {maintenance_results['tasks_completed'] / peak_results['tasks_completed'] * 100:.1f}%")
print(f"• Rush order adaptation: {rush_results['tasks_completed'] / peak_results['tasks_completed'] * 100:.1f}%")

# Recommendations
print(f"\nDigital Twin Recommendations:")
print(f"• Implement predictive maintenance scheduling based on crane usage patterns")
print(f"• Use real-time congestion prediction for dynamic task prioritization")
print(f"• Develop adaptive resource allocation algorithms for peak demand periods")
print(f"• Create alert prioritization system for critical task management")

### Why This Tier Exists vs Earlier Tiers

**Tier 5: Integrated Digital Twin** provides system-of-systems simulation with:
- **Real-time synchronization** between physical and virtual AS/RS systems
- **Predictive analytics** for demand forecasting and congestion prediction
- **What-if scenario testing** for disruption modeling and resilience analysis
- **Multi-system integration** connecting crane dynamics, storage states, and task queues

### Pros / Cons vs Alternative Approaches

**Advantages:**
- ✅ **Holistic simulation** - Captures system-wide interactions and emergent behaviors
- ✅ **Real-time monitoring** - Continuous state synchronization and alerting
- ✅ **Predictive capabilities** - Forecast demand and congestion before they occur
- ✅ **Scenario testing** - Safe what-if analysis without disrupting operations
- ✅ **Resilience analysis** - Test system performance under disruptions

**Disadvantages:**
- ❌ **Computational complexity** - High computational overhead for real-time simulation
- ❌ **Data requirements** - Extensive sensor network and data infrastructure
- ❌ **Model accuracy** - Predictions may have uncertainty and require calibration
- ❌ **Implementation complexity** - Complex system integration and maintenance
- ❌ **Resource requirements** - Significant computing and storage resources needed

### When to Use This Tier

**Use Digital Twin when:**
- System-wide understanding is required beyond individual optimization
- Real-time monitoring and alerting is critical for operations
- Predictive analytics can improve decision making and efficiency
- What-if scenario testing is valuable for planning and risk management
- System resilience under disruptions needs to be evaluated
- Multiple systems need to be integrated and coordinated

**Consider other tiers when:**
- Single-objective optimization is sufficient (use Tier 1, 2, or 3)
- Real-time performance is critical without complex system interactions (use Tier 2)
- Training data is limited for ML approaches (use Tier 4)
- Simple rule-based optimization is sufficient (use Tier 1 or 2)
- Computational resources are limited (use simpler approaches)